In [23]:
which python3

SyntaxError: invalid syntax (2784748077.py, line 1)

In [2]:
import pandas as pd
import numpy as np
import itertools
from sklearn.preprocessing import OneHotEncoder
from scipy import sparse
from lightfm import LightFM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
print("LightFM OK ✔️")


df = pd.read_csv("lol_dataset_challenger_grandmaster_clean.csv", sep=";")

print(df.shape)
print(df.columns.tolist()[:30])
df[['match_ids','teamId','championName','teamPosition','win','gameVersion']].head()
COLS = [
    'match_ids',        # ou 'match_ids'
    'teamId',
    'championName',
    'teamPosition',    # TOP/JUNGLE/MIDDLE/BOTTOM/UTILITY
    'win',
    'gameVersion',
    'gameStartTimestamp'  # optionnel mais utile pour time split
]

# adapte les noms si besoin
df0 = df[COLS].copy()
df0.head()
valid_pos = {'TOP','JUNGLE','MIDDLE','BOTTOM','UTILITY'}
df0 = df0[df0['teamPosition'].isin(valid_pos)].copy()
print(df0.shape)
df0['win01'] = df0['win'].astype(int) if df0['win'].dtype != 'O' else df0['win'].map({'Win':1,'Fail':0,'True':1,'False':0})
df0[['win','win01']].head()

def to_patch(v):
    if pd.isna(v): 
        return np.nan
    s = str(v)
    parts = s.split(".")
    return ".".join(parts[:2]) if len(parts) >= 2 else s

df0['patch'] = df0['gameVersion'].apply(to_patch)
df0[['gameVersion','patch']].head(10)

g = df0.groupby(['match_ids','teamId'])

team_size = g.size().rename('n_players')
n_roles = g['teamPosition'].nunique().rename('n_roles')

teams_ok = pd.concat([team_size, n_roles], axis=1).reset_index()
teams_ok = teams_ok[(teams_ok['n_players'] == 5) & (teams_ok['n_roles'] == 5)]

print("Teams ok:", teams_ok.shape)
teams_ok.head()

df1 = df0.merge(teams_ok[['match_ids','teamId']], on=['match_ids','teamId'], how='inner')
print(df1.shape)

df1['champ_role'] = df1['championName'].astype(str) + "_" + df1['teamPosition'].astype(str)

team_df = (
    df1.sort_values(['match_ids','teamId','teamPosition'])
       .groupby(['match_ids','teamId'])
       .agg(
           patch=('patch','first'),
           win=('win01','first'),
           start=('gameStartTimestamp','first'),
           champs=('champ_role', lambda x: list(x))
       )
       .reset_index()
)

team_df.head()


rows = []
for _, r in team_df.iterrows():
    champs = r['champs']
    for a, b in itertools.combinations(sorted(champs), 2):
        rows.append((r['match_ids'], r['teamId'], r['patch'], r['win'], a, b))

duos = pd.DataFrame(rows, columns=['match_ids','teamId','patch','win','a','b'])
print(duos.shape)
duos.head()

duo_stats = (
    duos.groupby(['a','b'])
        .agg(
            games=('win','size'),
            winrate=('win','mean')
        )
        .reset_index()
)

duo_stats = duo_stats[duo_stats['games'] >= 200].sort_values('winrate', ascending=False)
duo_stats.head(10)

##--------------------------------------------- duo winrate---------------------------------------------------------------------

champ_stats = (
    df1.groupby('champ_role')
       .agg(
           champ_games=('win01','size'),
           champ_winrate=('win01','mean')
       )
       .reset_index()
)
champ_stats.head()

duo_stats = duo_stats.merge(
    champ_stats.rename(columns={'champ_role':'a', 'champ_winrate':'wr_a'}),
    on='a', how='left'
)

duo_stats = duo_stats.merge(
    champ_stats.rename(columns={'champ_role':'b', 'champ_winrate':'wr_b'}),
    on='b', how='left'
)

duo_stats['expected_wr'] = (duo_stats['wr_a'] + duo_stats['wr_b']) / 2
duo_stats['delta_wr'] = duo_stats['winrate'] - duo_stats['expected_wr']
duo_synergy = (
    duo_stats[duo_stats['games'] >= 200]
    .sort_values('delta_wr', ascending=False)
)

duo_synergy.head(20)

##------------------------------------faux positifs eliminés (si deux champs sont op bah leur duo est aussi op useless a relever)---------

duo_synergy['score'] = duo_synergy['delta_wr'] * np.log1p(duo_synergy['games'])
duo_synergy = duo_synergy.sort_values('score', ascending=False)
duo_synergy.head(20)

duo_patch = (
    duos.groupby(['a','b','patch'])
        .agg(games=('win','size'), winrate=('win','mean'))
        .reset_index()
)

duo_patch_var = (
    duo_patch.groupby(['a','b'])
    .agg(
        patch_count=('patch','nunique'),
        wr_var=('winrate','var')
    )
    .reset_index()
)

duo_synergy = duo_synergy.merge(duo_patch_var, on=['a','b'], how='left')

duo_synergy['final_score'] = duo_synergy['score'] * (1 / (1 + duo_synergy['wr_var'].fillna(0)))
duo_synergy = duo_synergy.sort_values('final_score', ascending=False)
duo_synergy.head(20)

duo_synergy[['a','b','games','delta_wr','wr_var','final_score']].head(15)

#---------------------------apprentissage---------------------------------------------------------------------------

team_expanded = team_df.copy()
for i in range(5):
    team_expanded[f'champ_{i}'] = team_expanded['champs'].apply(lambda x: x[i])

X_cats = team_expanded[[f'champ_{i}' for i in range(5)]]
y = team_expanded['win'].values
enc = OneHotEncoder(handle_unknown='ignore')
X = enc.fit_transform(X_cats)

team_df.head()
team_df['champs'].apply(len).value_counts()

# 1) dictionnaire item_id pour chaque champ_role
all_items = sorted({cr for champs in team_df['champs'] for cr in champs})
item2id = {it:i for i,it in enumerate(all_items)}
id2item = {i:it for it,i in item2id.items()}

n_users = len(team_df)
n_items = len(all_items)

# 2) construire COO
rows = []
cols = []
data = []

for u, champs in enumerate(team_df['champs']):
    for cr in champs:
        rows.append(u)
        cols.append(item2id[cr])
        data.append(1.0)

interactions = sparse.coo_matrix((data, (rows, cols)), shape=(n_users, n_items)).tocsr()

y_win = team_df['win'].values.astype(np.float32)

interactions = interactions.tocoo()

rows = interactions.row
cols = interactions.col

# poids par interaction = 1 + alpha * win_de_l'équipe
alpha = 1.0

weights_data = []
for u in rows:
    w = 1.0 + alpha * y_win[u]
    weights_data.append(w)

weights = sparse.coo_matrix(
    (weights_data, (rows, cols)),
    shape=interactions.shape
)

model = LightFM(no_components=32, loss='bpr', random_state=42)

model.fit(
    interactions,
    sample_weight=weights,
    epochs=30,
    num_threads=4
)

item_emb = model.item_embeddings

print(item_emb.shape)

item_emb = model.item_embeddings  # (n_items, k)

def synergy_fm(a_cr, b_cr):
    ia = item2id.get(a_cr)
    ib = item2id.get(b_cr)
    if ia is None or ib is None:
        return np.nan
    return float(np.dot(item_emb[ia], item_emb[ib]))

print("Diana/Yasuo:", synergy_fm("Diana_JUNGLE", "Yasuo_MIDDLE"))
print("Kaisa/Ali:", synergy_fm("Kaisa_BOTTOM", "Alistar_UTILITY"))
print("Aphelios/Thresh:", synergy_fm("Aphelios_BOTTOM", "Thresh_UTILITY"))



def top_synergies_for(a_cr, topk=20):
    ia = item2id[a_cr]
    scores = item_emb @ item_emb[ia]   # dot avec tous
    best = np.argpartition(-scores, topk+1)[:topk+1]
    best = best[np.argsort(-scores[best])]
    out = []
    for j in best:
        if j == ia:
            continue
        out.append((id2item[j], float(scores[j])))
        if len(out) >= topk:
            break
    return out

model_pop = LightFM(no_components=32, loss='bpr', random_state=42)
model_pop.fit(interactions, epochs=30, num_threads=4)
emb_pop = model_pop.item_embeddings

model_win = LightFM(no_components=32, loss='bpr', random_state=42)
model_win.fit(interactions, sample_weight=weights, epochs=30, num_threads=4)
emb_win = model_win.item_embeddings

def synergy_win_minus_pop(a_cr, b_cr):
    ia = item2id.get(a_cr); ib = item2id.get(b_cr)
    if ia is None or ib is None:
        return np.nan
    s_win = float(np.dot(emb_win[ia], emb_win[ib]))
    s_pop = float(np.dot(emb_pop[ia], emb_pop[ib]))
    return s_win - s_pop

print("Azir + Maokai (win-pop):", synergy_win_minus_pop("Azir_MIDDLE", "Maokai_JUNGLE"))

def top_win_minus_pop_for(a_cr, topk=30):
    ia = item2id[a_cr]
    scores = (emb_win @ emb_win[ia]) - (emb_pop @ emb_pop[ia])
    best = np.argpartition(-scores, topk+1)[:topk+1]
    best = best[np.argsort(-scores[best])]
    out = []
    for j in best:
        if j == ia: 
            continue
        out.append((id2item[j], float(scores[j])))
        if len(out) >= topk:
            break
    return out

#-- win minus pop : si la popularité est high et le wr aussi -> pas du uplift, si la popularité est naze et le wr bien -> on remonte des 
#-- duos pas connus mais strong donc logique quon trouve pas diana pour yasuo ici car pop (high) moins wr (high) ~ 0 !

top_win_minus_pop_for("Azir_MIDDLE", 30)
top_synergies_for("Azir_MIDDLE", 20)

#--------------------------------------------------------------------------


duo_stats = (
    duos.groupby(['a','b'])
        .agg(
            games=('win','size'),
            winrate=('win','mean')
        )
        .reset_index()
)

print(duo_stats.shape)
duo_stats.head()


champ_rows = []
for _, r in team_df.iterrows():
    for cr in r['champs']:
        champ_rows.append((cr, r['win']))

champ_df = pd.DataFrame(champ_rows, columns=['champ_role','win'])

champ_stats = (
    champ_df.groupby('champ_role')
    .agg(
        champ_games=('win','size'),
        wr_a=('win','mean')
    )
    .reset_index()
)

duo_stats = duo_stats.merge(
    champ_stats.rename(columns={'champ_role':'a'}),
    on='a', how='left'
)

duo_stats = duo_stats.merge(
    champ_stats.rename(columns={'champ_role':'b', 'wr_a':'wr_b', 'champ_games':'champ_games_b'}),
    on='b', how='left'
)

duo_stats = duo_stats.rename(columns={'champ_games':'champ_games_a'})

duo_stats['expected_wr'] = (duo_stats['wr_a'] + duo_stats['wr_b']) / 2
duo_stats['delta_wr'] = duo_stats['winrate'] - duo_stats['expected_wr']


def dot_emb(emb, a_cr, b_cr):
    ia = item2id.get(a_cr)
    ib = item2id.get(b_cr)
    if ia is None or ib is None:
        return np.nan
    return float(np.dot(emb[ia], emb[ib]))

duo_stats['fm'] = duo_stats.apply(lambda r: dot_emb(emb_win, r['a'], r['b']), axis=1)
duo_stats['uplift'] = duo_stats.apply(lambda r: dot_emb(emb_win, r['a'], r['b']) - dot_emb(emb_pop, r['a'], r['b']), axis=1)

min_games = 30

duo_f = duo_stats[
    (duo_stats['games'] >= min_games)
    & (~duo_stats['uplift'].isna())
    & (~duo_stats['fm'].isna())
].copy()

print(duo_f.shape)

def zscore(s):
    return (s - s.mean()) / (s.std() + 1e-9)

duo_f['z_uplift'] = zscore(duo_f['uplift'])
duo_f['z_fm'] = zscore(duo_f['fm'])
duo_f['z_delta'] = zscore(duo_f['delta_wr'])

# pondérations discovery
duo_f['discovery_score'] = (
    0.5 * duo_f['z_uplift'] +
    0.3 * duo_f['z_fm'] +
    0.2 * duo_f['z_delta']
)

duo_f.sort_values('discovery_score', ascending=False).head(20)[
    ['a','b','games','winrate','delta_wr','uplift','fm','discovery_score']
]


pair_a = "Azir_MIDDLE"
pair_b = "Maokai_JUNGLE"

fm = dot_emb(emb_win, pair_a, pair_b)
pop = dot_emb(emb_pop, pair_a, pair_b)
uplift = fm - pop

print("fm:", fm)
print("pop:", pop)
print("uplift:", uplift)

#------------------------------------------------ MEILLEURS SYNERGIES A PARTIR DE DATA MICRO ---------------------------------------------

#--------------------------------------------------------------SYNERGIES MACRO------------------------------------------------------------

macro_cols = [
    'dragonKills',
    'baronKills',
    'damageDealtToObjectives',
    'damageDealtToTurrets',
    'turretKills',
    'inhibitorKills',
    'totalDamageDealtToChampions',
    'totalDamageTaken',
    'totalTimeCCDealt',
    'visionScore'
]

team_macro = (
    df.groupby(['match_ids','teamId'])
      .agg(
          dragonKills=('dragonKills','sum'),
          baronKills=('baronKills','sum'),
          damageDealtToObjectives=('damageDealtToObjectives','sum'),
          damageDealtToTurrets=('damageDealtToTurrets','sum'),
          turretKills=('turretKills','sum'),
          inhibitorKills=('inhibitorKills','sum'),
          totalDamageDealtToChampions=('totalDamageDealtToChampions','sum'),
          totalDamageTaken=('totalDamageTaken','sum'),
          totalTimeCCDealt=('totalTimeCCDealt','sum'),
          visionScore=('visionScore','sum'),
          gameDuration=('gameDuration','first')  # identique pour l’équipe
      )
      .reset_index()
)

print(team_macro.shape)
team_macro.head()

team_df = team_df.merge(team_macro, on=['match_ids','teamId'], how='left')

team_df['obj_control'] = (
    team_df['dragonKills']
    + 2 * team_df['baronKills']
    + 0.001 * team_df['damageDealtToObjectives']
)

team_df['siege'] = (
    team_df['turretKills']
    + 0.001 * team_df['damageDealtToTurrets']
)

team_df['teamfight'] = (
    team_df['totalDamageDealtToChampions']
    + 0.5 * team_df['totalDamageTaken']
)

team_df['pick_cc'] = team_df['totalTimeCCDealt']
team_df['vision'] = team_df['visionScore']
team_df['tempo'] = team_df['gameDuration']

style_cols = ['obj_control','siege','teamfight','pick_cc','vision','tempo']

scaler = StandardScaler()
team_df[style_cols] = scaler.fit_transform(team_df[style_cols])

rows = []
for _, r in team_df.iterrows():
    for cr in r['champs']:
        rows.append([cr] + list(r[style_cols]))

champ_style = pd.DataFrame(rows, columns=['champ_role'] + style_cols)

champ_style_vec = (
    champ_style.groupby('champ_role')[style_cols]
    .mean()
)

# ===== add extra combat structure features =====
team_df['frontline'] = team_df['totalDamageTaken']
team_df['dps'] = team_df['totalDamageDealtToChampions'] / team_df['gameDuration']
team_df['cc_density'] = team_df['totalTimeCCDealt'] / team_df['gameDuration']
team_df['vision_per_min'] = team_df['visionScore'] / team_df['gameDuration']
style_cols = [
    'obj_control','siege','teamfight',
    'pick_cc','vision','tempo',
    'frontline','dps','cc_density','vision_per_min'
]
scaler = StandardScaler()
team_df[style_cols] = scaler.fit_transform(team_df[style_cols])

rows = []
for _, r in team_df.iterrows():
    for cr in r['champs']:
        rows.append([cr] + list(r[style_cols]))

champ_style = pd.DataFrame(rows, columns=['champ_role'] + style_cols)

champ_style_vec = (
    champ_style.groupby('champ_role')[style_cols]
    .mean()
)

print("champ_style_vec cols:", champ_style_vec.columns.tolist())

def macro_complementarity(a_cr, b_cr):
    if a_cr not in champ_style_vec.index or b_cr not in champ_style_vec.index:
        return np.nan

    a = champ_style_vec.loc[a_cr]
    b = champ_style_vec.loc[b_cr]

    score = 0.0

    # tank + dps
    score += a['frontline'] * b['dps'] + b['frontline'] * a['dps']

    # CC + damage
    score += a['cc_density'] * b['dps'] + b['cc_density'] * a['dps']

    # vision/setup + tempo
    score += a['vision_per_min'] * b['tempo'] + b['vision_per_min'] * a['tempo']

    # objective control + teamfight
    score += a['obj_control'] * b['teamfight'] + b['obj_control'] * a['teamfight']

    return float(score)

def top_macro_complementarity(a_cr, topk=10):
    scores = []
    for b in champ_style_vec.index:
        if b == a_cr:
            continue
        scores.append((b, macro_complementarity(a_cr, b)))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    return scores[:topk]

# compter le nombre de games par champ_role
champ_counts = champ_style['champ_role'].value_counts()

# seuil (à ajuster : 50, 80, 100 selon ton dataset)
min_games = 200

valid_champs = champ_counts[champ_counts >= min_games].index

# filtrer
champ_style = champ_style[champ_style['champ_role'].isin(valid_champs)]

# reconstruire les vecteurs
champ_style_vec = (
    champ_style.groupby('champ_role')[style_cols]
    .mean()
)

print("nb champs après filtre:", len(champ_style_vec))

synergy_fm("

#--viens de montrer empiriquement :

# type de synergie apprise depuis la soloq 
# micro (mécanique : Diana/Yasuo) oui 
# duo botlane (Lucian/Nami)  oui 
# macro pro (Azir/Maokai)  non


LightFM OK ✔️


/tmp/ipykernel_11856/2865292928.py:12: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("lol_dataset_challenger_grandmaster_clean.csv", sep=";")


(181897, 140)
['match_ids', 'endOfGameResult', 'gameCreation', 'gameDuration', 'gameEndTimestamp', 'gameId', 'gameMode', 'gameName', 'gameStartTimestamp', 'gameType', 'gameVersion', 'mapId', 'platformId', 'queueId', 'tournamentCode', 'allInPings', 'assistMePings', 'assists', 'baronKills', 'basicPings', 'champExperience', 'champLevel', 'championId', 'championName', 'championTransform', 'commandPings', 'consumablesPurchased', 'damageDealtToBuildings', 'damageDealtToEpicMonsters', 'damageDealtToObjectives']
(177271, 7)
Teams ok: (35439, 4)
(177195, 9)
(354390, 6)
(667, 32)
Diana/Yasuo: 75.44124603271484
Kaisa/Ali: 34.75803756713867
Aphelios/Thresh: 72.01734924316406
Azir + Maokai (win-pop): -0.17598456144332886
(41617, 4)
(2828, 12)
fm: -0.9447305202484131
pop: -0.7687459588050842
uplift: -0.17598456144332886
(36378, 13)
champ_style_vec cols: ['obj_control', 'siege', 'teamfight', 'pick_cc', 'vision', 'tempo', 'frontline', 'dps', 'cc_density', 'vision_per_min']
nb champs après filtre: 196


[('Gangplank_TOP', 42.19097137451172),
 ('Nami_UTILITY', 41.86421585083008),
 ('Syndra_MIDDLE', 34.68059158325195),
 ('Smolder_BOTTOM', 32.47260665893555),
 ('Ambessa_TOP', 31.212528228759766),
 ('KSante_TOP', 29.218368530273438),
 ('Nautilus_UTILITY', 27.210174560546875),
 ('Caitlyn_BOTTOM', 23.867738723754883),
 ('LeeSin_JUNGLE', 23.45712661743164),
 ('Pyke_UTILITY', 22.043291091918945)]